# Objetivo
El dataset olist_order_items_dataset.csv contiene los siguientes campos

| Campo             |      descripcion   |     tipo de datos   | Acción
|-------------------|--------------------|---------------------|-------
|**order_id** | id_pedido |  object             
| **order_item_id** | índice del ítem dentro del pedido |  int64  |  
| **product_id** | id_producto |  object             
| **seller_id** | id_vendedor |  object              
| **shipping_limit_date**| fecha limite de envio |  object    **| Convertir en fecha
| **price**| precio | float64
| **freight_value** | valor del envio | float64
|


El objetivo de esta consulta es crear un DataFrame que informe en una sola línea la cantidad de un mismo producto que hay por orden (utilizando order_id & product_id) excluyendo el campo **order_item_id** (que tiene que ver con el índice que el producto tiene dentro del pedido)


# 1 Importar librerías

In [26]:
# Revisar coincidencias difusas entre cadenas de texto
#!pip install geopandas plotly shapely folium fuzzywuzzy python-Levenshtein
#!pip install pandas gradio matplotlib openai tqdm

# Util cuando ciertas advertencias no son críticas y o queremos que aparezcan en la salida
import warnings
warnings.filterwarnings('ignore')

# Manipulación y análisis de datos
import pandas as pd
import numpy as np

# Visualización de datos
#import matplotlib.pyplot as plt
#import matplotlib.dates as mdates
#import seaborn as sns
#import plotly.express as px

# Datos geoespaciales
#import geopandas as gpd
#from shapely.geometry import shape
#import folium

# Utilidades y manejo de archivos
import glob
import os

# Autorización para que GoogleColab acceda a Drive
from google.colab import drive
drive.mount('/content/drive')

# Opciones de visualización de pandas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Establecer paleta de colores global usando 'viridis'
#sns.set_palette("viridis")

# Para poder extraer un color base de esa paleta:
#base_color = sns.color_palette("viridis")[0]


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2 Función Análisis Exploratorio de Datos

In [27]:
# Definimos una función llamada 'exploracion_inicial' que puede llegar a recibir dos parametros
# 1- df         = DataFrame
# 2- tipo=None  = un argumento opcional para indicar si queremos una exploracion simple o completa

def exploracion_inicial(df, tipo=None):
    print("¿Cuántas filas y columnas hay en el conjunto de datos?")
    num_filas, num_columnas = df.shape
    print(f"\tHay {num_filas:,} filas y {num_columnas:,} columnas.")
    print('#' * 90)

    if tipo == 'simple':
        print ("¿Cuáles son las primeras dos filas del conjunto de datos?")
        display (df.head(2))

# Explicación de la función

# num_filas, num_columnas = df.shape
# df.shape                  = Me devuelve una tupla que contiene dos elementos:  Filas y Columnas
# num_filas, num_columnas   = como devuelve dos elementos podemos crear dos variables con la infomración que me esta devolviendo
#                             esta funcion  -> num_filas y num_columnas

# print(f"\tHay {num_filas:,} filas y {num_columnas:,} columnas.")
# f""           me permite incluir variables dentro de una cadena de texto
# \t            me añade un tabulador al inicio del texto
# {num_filas:,} la coma indica que, cuando haya números grandes debe mostrar la coma como separador de miles

#if tipo == 'simple':
#     display (df.head(2))
# si recibe el valor 'simple' me mostrara (display) de una forma visualmente más atractiva (que utilizando la función 'print')
# las dos primeras filas



    else:
        print("¿Cuáles son las primeras cinco filas del conjunto de datos?")
        display(df.head())

        print("¿Cuáles son las últimas cinco filas del conjunto de datos?")
        display(df.tail())

        print("¿Cómo puedes obtener una muestra aleatoria de filas del conjunto de datos?")
        display(df.sample(n=5))

        print("¿Cuáles son las columnas del conjunto de datos?")
        print("\n".join(f"\t- {col}" for col in df.columns))

        # "\n"                   une las cadenas generadas con saldos de linea
        # \t-                    añade un tabulador (\t) y un guion (-)
        # join(f"\t- {col}" for col in df.columns itero por todas las columnas del df


        print("¿Cuál es el tipo de datos de cada columna?")
        display(df.dtypes)

        print("¿Cuántas columnas hay de cada tipo de datos?")
        display(df.dtypes.value_counts())

        print("¿Cómo podríamos obtener información más completa sobre la estructura y el contenido del DataFrame?")
        df.info()

        print("¿Cuántos valores únicos tiene cada columna?")
        display(df.nunique())

        print("¿Cuáles son las estadísticas descriptivas básicas de todas las columnas?")
        display(df.describe(include='all').fillna(''))

        print("¿Hay valores nulos en el conjunto de datos?")
        display(df.isnull().sum())

        print("¿Cuál es la proporción de valores nulos en cada columna?")
        display((df.isnull().sum() / len(df) * 100).round(2))

    print('#' * 90)

# 3 Carga de datos

In [28]:
# products Dataset
df_products = pd.read_csv("/content/drive/MyDrive/nuclio/TFM NUCLIO/TFM Olist Ecommerce FINAL/01. Archivos CSV Limpios/olist_products_dataset_clean V2.csv")

In [29]:
# items Dataset
df_items = pd.read_csv("/content/drive/MyDrive/nuclio/TFM NUCLIO/TFM Olist Ecommerce FINAL/00. Datos/olist_order_items_dataset.csv")

# 4 Items Dataset
olist_order_items_dataset.csv

In [30]:
exploracion_inicial(df_items,'')

¿Cuántas filas y columnas hay en el conjunto de datos?
	Hay 112,650 filas y 7 columnas.
##########################################################################################
¿Cuáles son las primeras cinco filas del conjunto de datos?


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


¿Cuáles son las últimas cinco filas del conjunto de datos?


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72
112649,fffe41c64501cc87c801fd61db3f6244,1,350688d9dc1e75ff97be326363655e01,f7ccf836d21b2fb1de37564105216cc1,2018-06-12 17:10:13,43.00,12.79


¿Cómo puedes obtener una muestra aleatoria de filas del conjunto de datos?


...

¿Hay valores nulos en el conjunto de datos?


,0
order_id,0
order_item_id,0
product_id,0
seller_id,0
shipping_limit_date,0
price,0
freight_value,0


¿Cuál es la proporción de valores nulos en cada columna?


,0
order_id,0.0
order_item_id,0.0
product_id,0.0
seller_id,0.0
shipping_limit_date,0.0
price,0.0
freight_value,0.0


##########################################################################################


## 4.1 Formato fecha

In [31]:
# Damos formato fecha al campo 'shipping_limit_date'
df_items['shipping_limit_date'] = pd.to_datetime(df_items['shipping_limit_date'])

In [32]:
# confirmamos el cambio de formato de fecha
display (df_items.dtypes)

,0
order_id,object
order_item_id,int64
product_id,object
seller_id,object
shipping_limit_date,datetime64[ns]
price,float64
freight_value,float64


En la exploración incial que se hizo del dataset que no hay ninguna fila completamente igual a otra en todas sus columnas.  

Sin embargo en el campo **order_id** se observa que la cantidad de ordenes es mayor a los valores únicos producto de que, en una misma orden, puede haber varios productos (con el mismo o diferente id)

| Campo             |      count         |     unique
|-------------------|--------------------|---------------------
|**order_id**       |    112650          |  98666

Se hace una exploración de los duplicados aislando el campo **order_item_id** (índice) para confirmar si en producto puede estar en el mismo pedido más de una vez.

In [33]:
# Contar duplicados sin considerar la columna 'order_item_id'
duplicados_sin_item_id = df_items.drop(columns=['order_item_id']).duplicated().sum()
duplicados_sin_item_id


np.int64(10225)

Confirmamos que, sin considerar la columna **order_item_id**, existen 10225 con el mismo id de producto.  Es decir, que una misma orden puede tener + de 1 producto con el mismo id.

Con el siguiente código se visualiza uno de los pedidos que tienen más de un mismo código de producto.

In [34]:
# Obtener los order_id que tienen duplicados (sin contar order_item_id)
duplicados_sin_item_id_df = df_items[df_items.drop(columns=['order_item_id']).duplicated()]
pedidos_duplicados = duplicados_sin_item_id_df['order_id'].unique()

# Seleccionar el primer pedido duplicado y mostrar todas sus filas
pedido_ejemplo = pedidos_duplicados[2]
df_items[df_items['order_id'] == pedido_ejemplo]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
42,001ab0a7578dd66cd4b0a71f5b6e1e41,1,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63
43,001ab0a7578dd66cd4b0a71f5b6e1e41,2,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63
44,001ab0a7578dd66cd4b0a71f5b6e1e41,3,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63


**df_items_modificado**

Se crea un nuevo DataFrame sin el campo **order_item_id** con un campo adicional **qty_item_id** que contiene cuantas veces se repite un mismo producto en la orden.
De esta forma, en una misma línea, estara la cantidad de articulos que tienen el mismo id, su precio unitario y el costo del envío de ese articulo.

In [35]:
# Crear una nueva clave combinada: order_id + product_id
df_items['clave'] = df_items['order_id'] + '_' + df_items['product_id']

# Calcular la cantidad de veces que se repite cada combinación order_id + product_id
df_items['qty_item_id'] = df_items.groupby('clave')['clave'].transform('count')

# Seleccionar las columnas requeridas
df_items_modificado = df_items[[
    'order_id', 'product_id', 'seller_id',
    'shipping_limit_date', 'price', 'freight_value', 'qty_item_id'
]]

# Eliminar posibles duplicados si la combinación order_id + product_id se repite
df_items_modificado = df_items_modificado.drop_duplicates(subset=['order_id', 'product_id'])



Confirmamos las modificaciones

In [36]:
display (df_items_modificado.dtypes)

,0
order_id,object
product_id,object
seller_id,object
shipping_limit_date,datetime64[ns]
price,float64
freight_value,float64
qty_item_id,int64


Confirmamos que ninguna fila completamente igual a otra en todas sus columnas.

## 4.2 Exportacion df_items_limpio_with_qty.csv

In [37]:
# Vemos como quedaría la tabla sín el campo 'order_item_id'
df_items_modificado[df_items_modificado['order_id'] == '001ab0a7578dd66cd4b0a71f5b6e1e41']

,order_id,product_id,seller_id,shipping_limit_date,price,freight_value,qty_item_id
42,001ab0a7578dd66cd4b0a71f5b6e1e41,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63,3


In [38]:
df_items_modificado.to_csv('/content/drive/MyDrive/nuclio/TFM NUCLIO/TFM Olist Ecommerce FINAL/01. Archivos CSV Limpios/df_items_clean_with_qty.csv', index=False)